# ⚖️ 台灣司法官律師國考 模型微調 (Kaggle GPU 專用極速版)
本 Notebook 專為 **Kaggle GPU (GPU T4 x2 或 P100)** 最佳化，提供完整的 Unsloth 4-bit 顯存優化與多源資料加權微調架構。

### ⚠️ 執行前 Kaggle 重要設定（必做！）：
1. **開啟 GPU 加速**：右側邊欄 `Settings` -> `Accelerator` 選擇 **`GPU T4 x2`**（或 P100）。
2. **開啟網路權限**：右側邊欄 `Settings` -> `Internet` 務必勾選 **`Internet on`**（下載模型與套件必備）。
3. **掛載訓練資料集**：右側邊欄點擊 `+ Add Input` 上傳您的 `law_exam_balanced_weighted_sft.jsonl`（或使用下方自動下載方式）。

In [ ]:
# 1. 安裝 Kaggle 專用 Unsloth 與依賴套件
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 指定主 GPU 進行極速微調

!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo datasets trl peft bitsandbytes gdown

import torch
print('✅ GPU 設備狀態：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '未偵測到 GPU，請在 Settings 開啟 GPU！')


In [ ]:
import os, re, glob
from pathlib import Path

#@title 🚀 資料集路徑配置（支援從 Google Drive 調用或 Kaggle 本地掛載）
#@markdown 【隱私安全警示】：若資料集包含個人手寫筆記或考生隱私，請注意 Google Drive「知道連結者皆可檢視」為公開分享。若需嚴格保密，強烈推薦使用 Kaggle 右側 [+ Add Input] 掛載為私人 Private Dataset。
#@markdown 若欲使用 Google Drive 下載，請在此填入檔案分享連結或 File ID；若欲使用 Kaggle Input 本地掛載，請保留留空：
GDRIVE_URL_OR_ID = "" #@param {type:"string"}
ALLOW_SMOKE_TEST_IF_NO_DATA = False #@param {type:"boolean"} 僅供無資料時跑語法測試

DATA_PATH = None

# 分支 1：使用者指定 Google Drive 來源（嚴格來源隔離，失敗絕不隱式切換）
if GDRIVE_URL_OR_ID.strip():
    raw_val = GDRIVE_URL_OR_ID.strip()
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", raw_val)
    file_id = match.group(1) if match else raw_val
    DATA_PATH = "/kaggle/working/law_exam_balanced_weighted_sft.jsonl"
    print(f'☁️ 正在自 Google Drive 下載微調資料集 (ID: {file_id})...')
    !gdown --id {file_id} -O {DATA_PATH}
    if not os.path.exists(DATA_PATH) or os.path.getsize(DATA_PATH) < 1000000:
        !gdown "{raw_val}" -O {DATA_PATH} --fuzzy

    # 嚴格校驗：檔案大小檢查 (>1MB) + 逐行抽樣 JSONL 語法與首末行完整性防護
    if os.path.exists(DATA_PATH) and os.path.getsize(DATA_PATH) > 1000000:
        import json
        valid_rows = 0
        try:
            with open(DATA_PATH, "r", encoding="utf-8") as f_chk:
                for idx, line in enumerate(f_chk):
                    l_str = line.strip()
                    if l_str:
                        if idx < 5 or idx % 10000 == 0:
                            json.loads(l_str)
                        valid_rows += 1
            if valid_rows > 1000:
                print(f'✅ 成功調用資料集並通過完整性校驗！有效樣本：{valid_rows:,} 筆，大小：{os.path.getsize(DATA_PATH)/(1024*1024):.2f} MB')
            else:
                raise ValueError(f"有效資料行數過少 ({valid_rows} 筆)，可能為截斷檔案")
        except Exception as chk_err:
            if os.path.exists(DATA_PATH):
                os.remove(DATA_PATH)
            raise RuntimeError(f"❌ Google Drive 資料集下載或校驗失敗 ({chk_err})！已清除異常暫存檔，終止執行以防止誤訓殘缺資料。") from chk_err
    else:
        if os.path.exists(DATA_PATH):
            os.remove(DATA_PATH)
        raise RuntimeError("❌ Google Drive 下載失敗（檔案大小異常或無權限存取）！為維護訓練純度，系統已終止執行。請確認檔案連結分享權限或改採 Kaggle Input 本地掛載。")

# 分支 2：使用者未指定 Drive 連結，搜尋 /kaggle/input (透過右側 + Add Input 本地掛載)
else:
    found_datasets = glob.glob("/kaggle/input/**/law_exam*.jsonl", recursive=True)
    if found_datasets:
        for p in found_datasets:
            if "weighted" in p.lower():
                DATA_PATH = p
                break
        if not DATA_PATH:
            DATA_PATH = found_datasets[0]
        print(f'✅ 自動偵測到 Kaggle Input 本地資料集：{DATA_PATH}')

# 防護機制：無任何可用資料集時阻斷
if not DATA_PATH:
    if ALLOW_SMOKE_TEST_IF_NO_DATA:
        DATA_PATH = "/kaggle/working/sample_data.jsonl"
        import json
        sample_item = {
            "id": "smoke_test_001",
            "source_type": "smoke_test_sample",
            "instruction": "請以 IRAC 架構解答刑法生命法益爭點。",
            "input": "醫師乙趁胎兒頭部未露出產道故意致其死亡，論以何罪？",
            "output": "採陣痛開始說（王皇玉、林山田），自然產以規則陣痛開始為人之始期，應成立刑法第271條普通殺人罪。"
        }
        with open(DATA_PATH, "w", encoding="utf-8") as f_s:
            for _ in range(50):
                f_s.write(json.dumps(sample_item, ensure_ascii=False) + "
")
        print('=' * 75)
        print('⚠️ [SMOKE TEST 模式] 使用 50 筆測試樣本僅供驗證 GPU 語法流程，絕非正式訓練！')
        print('=' * 75)
    else:
        raise FileNotFoundError(
            '❌ 未偵測到微調資料集！\n'
            '【雲端下載方式】：請在上方 GDRIVE_URL_OR_ID 貼上 Google Drive 檔案共用連結或 File ID；\n'
            '【私有保密推薦】：於 Kaggle 右側點擊 [+ Add Input] -> 上傳 law_exam_balanced_weighted_sft.jsonl 作為 Private Dataset；\n'
            '【語法測試】：若僅欲驗證流程，請勾選 ALLOW_SMOKE_TEST_IF_NO_DATA。'
        )

In [ ]:
# 3. 載入基底開源模型或既有 LoRA 權重 (跨 Session 增量微調 Continual SFT / Warm Start)
# 說明：載入既有 LoRA 僅繼承模型適配層權重 (Weights Only)，並以全新優化器展開第二階段增量訓練。
from unsloth import FastLanguageModel
import os, glob, zipfile
from pathlib import Path

max_seq_length = 2048
load_in_4bit = True

#@title 接續先前已訓練 LoRA 權重配置 (可選)
#@markdown 若欲在先前微調過的成果上繼續增量訓練，請填寫 LoRA 目錄或 Zip 檔案路徑：
EXISTING_LORA_PATH = "" #@param {type:"string"}

# 自動掃描 /kaggle/input 是否有先前上傳的 LoRA 轉接器
if not EXISTING_LORA_PATH.strip():
    found_loras = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if found_loras:
        EXISTING_LORA_PATH = str(Path(found_loras[0]).parent)
        print(f'🔍 自動在 Kaggle Input 偵測到既有 LoRA 權重：{EXISTING_LORA_PATH}')

if EXISTING_LORA_PATH.strip() and os.path.exists(EXISTING_LORA_PATH.strip()):
    lora_target = EXISTING_LORA_PATH.strip()
    if lora_target.endswith(".zip"):
        extract_dir = "/kaggle/working/previous_lora"
        with zipfile.ZipFile(lora_target, "r") as z:
            z.extractall(extract_dir)
        lora_target = extract_dir
    print(f'🔄 正在載入既有 LoRA 權重以支援【增量持續微調 (Continual Fine-Tuning)】：{lora_target}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = lora_target,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_training(model)
    print('✅ 既有 LoRA 權重載入完成，已重啟可訓練梯度，準備接續增量微調！')
else:
    print('🚀 載入官方開源基底模型：unsloth/Qwen2.5-7B-Instruct-bnb-4bit')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = load_in_4bit,
    )
    # 配置高效能 LoRA 轉接器
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    print('✅ Qwen2.5-7B-Instruct 模型與 LoRA 介面全新初始化成功！')


In [ ]:
# 4. 嚴格 5 欄位特徵綱要載入、防止重複過採樣與 Prompt 格式化
from datasets import load_dataset, Features, Value
from trl import SFTTrainer
from transformers import TrainingArguments
import random

# 嚴格定義 5 大標準欄位，保證 100% 綱要相容 (徹底杜絕 CastError)
sft_features = Features({
    "id": Value("string"),
    "source_type": Value("string"),
    "instruction": Value("string"),
    "input": Value("string"),
    "output": Value("string"),
})

# 嚴格模式載入：不進行靜默降級，確保所有樣本均符合 5 欄位規範
dataset = load_dataset('json', data_files=DATA_PATH, split='train', features=sft_features)
print(f'✅ 成功以嚴格綱要載入原始樣本：{len(dataset):,} 筆')

#@title 來源權重動態平衡防護 (防止已加權檔案遭遇二次過採樣)
is_already_weighted = any(k in str(DATA_PATH).lower() for k in ["weighted", "balanced"])

if is_already_weighted:
    print('ℹ️ 偵測到已使用平衡加權庫 (law_exam_balanced_weighted_sft.jsonl)！')
    print('   已自動跳過第二輪重複過採樣，避免資料集膨脹至 650 萬筆導致 RAM OOM！')
else:
    ENABLE_DYNAMIC_WEIGHTING = True #@param {type:"boolean"}
    NOTE_WEIGHT = 500 #@param {type:"integer"} 手寫筆記權重倍數
    EXAM_WEIGHT = 50  #@param {type:"integer"} 司律真題權重倍數
    COURT_WEIGHT = 15 #@param {type:"integer"} 憲法與裁判權重倍數
    
    if ENABLE_DYNAMIC_WEIGHTING and "source_type" in dataset.column_names:
        print('⚖️ 正在對原始未加權語料套用過採樣 (Weighted Oversampling)...')
        sts = dataset["source_type"]
        idx_pool = []
        n_cnt, e_cnt, c_cnt, s_cnt = 0, 0, 0, 0
        for i, s in enumerate(sts):
            st = str(s).lower()
            if "handwritten" in st or "user_note" in st:
                idx_pool.extend([i] * NOTE_WEIGHT)
                n_cnt += NOTE_WEIGHT
            elif "exam" in st or "coursework" in st or "mcq" in st:
                idx_pool.extend([i] * EXAM_WEIGHT)
                e_cnt += EXAM_WEIGHT
            elif "constitutional" in st or "judgment" in st or "rule" in st or "theories" in st:
                idx_pool.extend([i] * COURT_WEIGHT)
                c_cnt += COURT_WEIGHT
            else:
                idx_pool.append(i)
                s_cnt += 1
        random.seed(3407)
        random.shuffle(idx_pool)
        dataset = dataset.select(idx_pool)
        total = len(dataset)
        print(f'✅ 動態加權完成！加權後訓練總樣本數：{total:,} 筆')
        print(f'   - 📝 手寫筆記佔比：{n_cnt/total*100:.2f}% ({n_cnt:,} 次曝光)')
        print(f'   - 🎓 司律真題/講義佔比：{e_cnt/total*100:.2f}% ({e_cnt:,} 次曝光)')
        print(f'   - 🏛️ 憲法/裁判規準佔比：{c_cnt/total*100:.2f}% ({c_cnt:,} 次曝光)')

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = (
            f"<|im_start|>system\n你是一位具備多年命題閱卷經驗之台灣司法官律師考試閱卷委員與法學權威。<|im_end|>\n"
            f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        texts.append(text)
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'成功載入並格式化 {len(dataset):,} 筆微調樣本！')


In [ ]:
# 5. 啟動 SFT 訓練引擎 (支援同環境 Checkpoint 完整斷點續訓)
# 說明：RESUME_FROM_CHECKPOINT 會自動恢復模型權重、AdamW 優化器狀態、學習率排程及步數，實現零損失接續。
from unsloth import is_bfloat16_supported
import os, glob

#@title 訓練步數與接續訓練設定
#@markdown 目標總訓練步數 (若接續訓練，設為大於上次步數的值，例如上次 200 步，這次設 500 或 1000 步)：
max_training_steps = 500 #@param {type:"integer"}
#@markdown 是否自動從上次未跑完或已存檔之 Checkpoint 接續訓練：
RESUME_FROM_CHECKPOINT = True #@param {type:"boolean"}

output_dir = "/kaggle/working/outputs"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = max_training_steps,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        save_steps = 100, # 每 100 步自動存檔一次 Checkpoint
        save_total_limit = 3, # 最多保留 3 個最新 Checkpoint
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = output_dir,
    ),
)

# 智慧辨識可接續之 Checkpoint
checkpoint_to_resume = None
if RESUME_FROM_CHECKPOINT and os.path.exists(output_dir):
    ckpts = sorted(
        glob.glob(f"{output_dir}/checkpoint-*"),
        key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0
    )
    if ckpts:
        checkpoint_to_resume = ckpts[-1]
        print(f'🔄 偵測到既有 Checkpoint，將自動自 【{checkpoint_to_resume}】 接續訓練至第 {max_training_steps} 步！')
    else:
        print('ℹ️ 未找到既有 Checkpoint，將從 Step 0 啟動全新訓練。')
else:
    print('🚀 啟動全新訓練...')

print('🔥 開始執行訓練流程...')
trainer_stats = trainer.train(resume_from_checkpoint=checkpoint_to_resume)
print('🎉 訓練完成！統計資訊：', trainer_stats)


In [ ]:
# 6. 儲存 LoRA 權重並打包為 ZIP 供一鍵下載
import shutil

SAVE_DIR = "/kaggle/working/qwen_law_evaluator_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'✅ LoRA 權重已儲存至：{SAVE_DIR}')

# 打包為 zip 檔，方便從 Kaggle 右側 Output 面板直接點擊 Download
zip_path = "/kaggle/working/qwen_law_evaluator_lora.zip"
shutil.make_archive("/kaggle/working/qwen_law_evaluator_lora", 'zip', SAVE_DIR)
print(f'📦 已打包下載檔：{zip_path} (您可直接在右側 Output 面板點擊下載！)')

# (選填) 若有 Hugging Face 帳號，亦可直接 1 鍵推送到雲端 Hub：
# HUGGINGFACE_TOKEN = "hf_xxxx" #@param {type:"string"}
# HF_REPO_NAME = "your_name/qwen_law_evaluator_lora" #@param {type:"string"}
# if HUGGINGFACE_TOKEN.strip():
#     model.push_to_hub(HF_REPO_NAME, token=HUGGINGFACE_TOKEN)
#     tokenizer.push_to_hub(HF_REPO_NAME, token=HUGGINGFACE_TOKEN)
#     print(f'🚀 已成功推送到 Hugging Face：https://huggingface.co/{HF_REPO_NAME}')
